# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is FAIR and described via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Display summary metadata
meta = dataset.metadata
print(f"Dataset name: {meta.name}\nDescription: {meta.description}")
print(f"Published: {getattr(meta, 'datePublished', 'n/a')}\nVersion: {getattr(meta, 'version', 'n/a')}")

## 2. Data Overview

Review available record sets (tables) and their fields (columns), referencing their `@id` according to the schema.

In [ ]:
# List record sets and fields by their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Available record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"- RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(unnamed)')}")
    print(f"  Description: {rs.get('description','')}")
    if 'fields' in rs:
        print(f"  Fields:")
        for field in rs['fields']:
            if isinstance(field, dict):
                fid = field.get('@id')
                print(f"    - {fid}")
            else:
                print(f"    - {field}")
    print()
if not record_sets:
    print("No top-level record set found in metadata; trying via the RDF graph.")
    # Sometimes Croissant puts record sets under .record_set
    if hasattr(dataset.metadata, 'record_set'):
        record_sets = dataset.metadata.record_set
        for rs in record_sets:
            print(f"- RecordSet: {rs['@id']}")

## 3. Data Extraction

Load data from a given record set into a DataFrame for further analysis. All references use the entity `@id` values from the dataset schema.

In [ ]:
# Find all record set IDs:
record_set_ids = []
for rs in dataset.metadata.record_sets:
    record_set_ids.append(rs['@id'])
# (If no .record_sets, try .record_set)
if not record_set_ids and hasattr(dataset.metadata, 'record_set'):
    for rs in dataset.metadata.record_set:
        record_set_ids.append(rs['@id'])
print("Record set @id list: ", record_set_ids)

dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame with {df.shape[0]} rows and {df.shape[1]} columns.")
        print(f"Fields (columns):\n{df.columns.tolist()}")
        display(df.head(3))
    except Exception as e:
        print(f"  Failed to load records: {e}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing: filtering, normalization, and grouping using a numeric field from a record set, referencing fields by their `@id`.

In [ ]:
# For demonstration, use first non-empty DataFrame and detect numeric fields.
for recset_id, df in dataframes.items():
    if not df.empty:
        # List numeric fields by @id (column name)
        numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
        if not numeric_cols:
            # Try to coerce all fields with number-like names to numeric if possible
            for col in df.columns:
                try:
                    df[col] = pd.to_numeric(df[col])
                except Exception:
                    continue
            numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
        if not numeric_cols:
            print(f"No numeric fields found in {recset_id}.")
            continue
        numeric_field_id = numeric_cols[0]
        print(f"Using numeric field: {numeric_field_id} from record set: {recset_id}")
        threshold = df[numeric_field_id].mean() if len(df) > 0 else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered {filtered_df.shape[0]} records where {numeric_field_id} > mean ({threshold:.2f})")
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Sample normalized data:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        # Find a categorical/group field
        group_candidates = [c for c in df.columns if c != numeric_field_id]
        group_field = group_candidates[0] if group_candidates else None
        if group_field is not None:
            print(f"Grouping by: {group_field} (@id field)")
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean values:")
            display(grouped_df.head())
        # Exit after the first successful run
        break

## 5. Visualization

Visualize the distribution of one numeric field and, if possible, compare grouped means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot numeric field distribution (using previous section variables if available)
try:
    sns.set(style="whitegrid")
    fig, ax = plt.subplots(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id} in {recset_id}")
    plt.show()
    # If group_field available, show grouped bar plot
    if group_field is not None and group_field in grouped_df.columns:
        plt.figure(figsize=(8,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} grouped by {group_field}")
        plt.xticks(rotation=45)
        plt.show()
except Exception as e:
    print(f"Visualization not available: {e}")

## 6. Conclusion

- Loaded the FAIR<sup>2</sup> dataset via its Croissant schema using `mlcroissant`.
- Enumerated the record sets and fields using `@id` identifiers per best practice.
- Extracted a full record set to a DataFrame, performed filtering, normalization, and grouping operations.
- Visualized the numeric field distribution and summarized means by group identifier when available.

> This notebook can be further extended for task-specific analyses, with additional domain knowledge applied to protein abundance and modification data in the context of extracellular vesicle mass spectrometry.
